# 08 — Multi-Backbone Ternary Classification (BanglaSarc3)

**Goal:** Train 5 transformer backbones on BanglaSarc3 ternary (3-class: Sarcastic, Non-Sarcastic, Neutral).

**Backbones:**
1. `csebuetnlp/banglabert` — Best binary backbone
2. `google/muril-base-cased` — Best of the new four
3. `xlm-roberta-base` — Strong multilingual baseline
4. `bert-base-multilingual-cased` — Classic multilingual
5. `sagorsarker/bangla-bert-base` — Alternative Bengali BERT

**Protocol:** epochs=3, lr=2e-5, batch=8, max_len=128, weighted loss for class imbalance  
**Disk-safe:** Same /kaggle/tmp strategy as notebook 07

In [ ]:
!pip install --upgrade transformers huggingface_hub -q

In [ ]:
import os, sys, json, time, warnings, gc, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

# ── Paths (same layout as notebook 07) ──
if os.path.exists('/kaggle/working'):
    REPO_ROOT = Path('/kaggle/working/Sarcasm_detection')
    BIG_TMP = Path('/kaggle/tmp')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '00_admin').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    BIG_TMP = REPO_ROOT / '_tmp'

PROJECT     = REPO_ROOT
SPLIT_DIR   = PROJECT / '01_data' / 'interim' / 'splits'
TABLE_DIR   = PROJECT / '04_outputs' / 'tables'
PRED_DIR    = PROJECT / '04_outputs' / 'predictions'
LOG_DIR     = PROJECT / '04_outputs' / 'run_logs'
REPORT_DIR  = PROJECT / '04_outputs' / 'reports'

for d in [TABLE_DIR, PRED_DIR, LOG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR   = BIG_TMP / 'hf_cache'
TEMP_TRAIN_DIR = BIG_TMP / 'train_cache'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def disk_free_gb(path='/'):
    try:
        return shutil.disk_usage(path).free / 1e9
    except FileNotFoundError:
        return shutil.disk_usage('.').free / 1e9

def clear_hf_cache():
    if HF_CACHE_DIR.exists():
        shutil.rmtree(HF_CACHE_DIR, ignore_errors=True)
        HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"  Cleared HF cache")

print(f"Project root: {PROJECT}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")
if os.path.exists('/kaggle/working'):
    print(f"Disk free (working): {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"Disk free (tmp):     {disk_free_gb(BIG_TMP):.1f} GB")
    clear_hf_cache()

In [ ]:
# ── Configuration ──

# All 5 backbones (including BanglaBERT which was only binary in nb05)
BACKBONES = {
    'banglabert':     'csebuetnlp/banglabert',
    'muril':          'google/muril-base-cased',
    'xlm-roberta':    'xlm-roberta-base',
    'mbert':          'bert-base-multilingual-cased',
    'sagorsarker-bb': 'sagorsarker/bangla-bert-base',
}

# Ternary dataset only
TERNARY_SPLITS = {
    'train': SPLIT_DIR / 'banglasarc3_ternary_train.csv',
    'val':   SPLIT_DIR / 'banglasarc3_ternary_val.csv',
    'test':  SPLIT_DIR / 'banglasarc3_ternary_test.csv',
}

NUM_LABELS    = 3
LABEL_NAMES   = ['Non-Sarcastic', 'Neutral', 'Sarcastic']  # Adjust if your label mapping differs
MAX_LENGTH    = 128
EPOCHS        = 3
BATCH_SIZE    = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
SEED          = 42
METRIC_FOR_BEST = 'macro_f1'

TEXT_COL  = 'text'
LABEL_COL = 'label_original'  # Ternary labels: 0, 1, 2

print(f"Backbones: {list(BACKBONES.keys())}")
print(f"Total runs: {len(BACKBONES)}")

In [ ]:
# ── Dataset class & helpers ──

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length',
                                   max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    accuracy_score(labels, preds),
        'macro_f1':    f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'macro_precision': precision_score(labels, preds, average='macro'),
        'macro_recall':    recall_score(labels, preds, average='macro'),
    }


def load_split(csv_path):
    df = pd.read_csv(csv_path)
    return df[TEXT_COL].astype(str).tolist(), df[LABEL_COL].astype(int).tolist()


# ── Sanity check + compute class weights ──
train_texts, train_labels = load_split(TERNARY_SPLITS['train'])
val_texts, val_labels     = load_split(TERNARY_SPLITS['val'])
test_texts, test_labels   = load_split(TERNARY_SPLITS['test'])

label_counts = Counter(train_labels)
print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
print(f"Label distribution (train): {dict(sorted(label_counts.items()))}")
print(f"Unique labels: {sorted(set(train_labels))}")

# Compute inverse-frequency class weights
total = sum(label_counts.values())
n_classes = len(label_counts)
class_weights = [total / (n_classes * label_counts[i]) for i in range(n_classes)]
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32)
print(f"Class weights: {[f'{w:.3f}' for w in class_weights]}")

In [ ]:
# ── Weighted-loss Trainer ──

class WeightedTrainer(Trainer):
    """Trainer that uses class weights in the cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weight = CLASS_WEIGHTS.to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ── Training function ──

def train_ternary(model_tag, model_name):
    print(f"\n{'='*70}")
    print(f"  {model_tag} x banglasarc3_ternary")
    print(f"  Model: {model_name}")
    if os.path.exists('/kaggle/working'):
        print(f"  Disk free (tmp): {disk_free_gb(BIG_TMP):.1f} GB | working: {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"{'='*70}")
    t_start = time.time()

    # ── Load data ──
    tr_texts, tr_labels = load_split(TERNARY_SPLITS['train'])
    vl_texts, vl_labels = load_split(TERNARY_SPLITS['val'])
    te_texts, te_labels = load_split(TERNARY_SPLITS['test'])

    # ── Tokenizer & model ──
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=str(HF_CACHE_DIR))
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_LABELS, ignore_mismatched_sizes=True,
        cache_dir=str(HF_CACHE_DIR)
    )

    # ── Build datasets ──
    train_ds = SarcasmDataset(tr_texts, tr_labels, tokenizer, MAX_LENGTH)
    val_ds   = SarcasmDataset(vl_texts, vl_labels, tokenizer, MAX_LENGTH)
    test_ds  = SarcasmDataset(te_texts, te_labels, tokenizer, MAX_LENGTH)

    # ── Temp dir ──
    run_tmp = TEMP_TRAIN_DIR / f"{model_tag}_ternary"
    if run_tmp.exists():
        shutil.rmtree(run_tmp)
    run_tmp.mkdir(parents=True, exist_ok=True)

    # ── Training args ──
    training_args = TrainingArguments(
        output_dir=str(run_tmp),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model=METRIC_FOR_BEST,
        greater_is_better=True,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to='none',
        dataloader_num_workers=2,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # ── Predict val ──
    val_output = trainer.predict(val_ds)
    val_metrics = val_output.metrics
    val_preds = np.argmax(val_output.predictions, axis=-1)
    val_probs = torch.softmax(torch.tensor(val_output.predictions), dim=-1).numpy()
    print(f"  Val  — Acc: {val_metrics['test_accuracy']:.4f} | Macro-F1: {val_metrics['test_macro_f1']:.4f}")

    # ── Predict test ──
    test_output = trainer.predict(test_ds)
    test_metrics = test_output.metrics
    test_preds = np.argmax(test_output.predictions, axis=-1)
    test_probs = torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()
    print(f"  Test — Acc: {test_metrics['test_accuracy']:.4f} | Macro-F1: {test_metrics['test_macro_f1']:.4f}")

    # ── Save predictions ──
    for split_name, texts, labels, preds, probs in [
        ('test', te_texts, te_labels, test_preds, test_probs),
        ('val',  vl_texts, vl_labels, val_preds,  val_probs),
    ]:
        pred_df = pd.DataFrame({
            'text': texts, 'true_label': labels, 'pred_label': preds,
            'prob_0': probs[:, 0], 'prob_1': probs[:, 1], 'prob_2': probs[:, 2],
        })
        pred_df.to_csv(
            PRED_DIR / f"08_{model_tag}_banglasarc3_ternary_{split_name}_predictions.csv",
            index=False
        )

    # ── Confusion matrix ──
    cm = confusion_matrix(te_labels, test_preds).tolist()
    t_elapsed = time.time() - t_start

    result = {
        'model_tag': model_tag, 'model_name': model_name,
        'dataset': 'banglasarc3_ternary',
        'val_accuracy':    val_metrics['test_accuracy'],
        'val_macro_f1':    val_metrics['test_macro_f1'],
        'val_weighted_f1': val_metrics['test_weighted_f1'],
        'test_accuracy':    test_metrics['test_accuracy'],
        'test_macro_f1':    test_metrics['test_macro_f1'],
        'test_weighted_f1': test_metrics['test_weighted_f1'],
        'test_macro_precision': test_metrics['test_macro_precision'],
        'test_macro_recall':    test_metrics['test_macro_recall'],
        'confusion_matrix': json.dumps(cm),
        'class_weights': json.dumps([round(w, 4) for w in class_weights]),
        'train_samples': len(tr_texts),
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE,
        'max_length': MAX_LENGTH, 'time_seconds': round(t_elapsed, 1),
    }

    # ── Cleanup ──
    del model, trainer, train_ds, val_ds, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if run_tmp.exists():
        shutil.rmtree(run_tmp)
    clear_hf_cache()
    print(f"  Cleaned up. Time: {t_elapsed/60:.1f} min")

    return result

In [ ]:
# ── Run all 5 backbones (with resume support) ──

all_results = []
summary_path = TABLE_DIR / '08_multi_backbone_ternary_summary.csv'

if summary_path.exists():
    prev_df = pd.read_csv(summary_path)
    all_results = prev_df.to_dict('records')
    done_tags = set(prev_df['model_tag'])
    print(f"Resuming: {len(done_tags)} runs already completed")
else:
    done_tags = set()

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
    TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
clear_hf_cache()

total = len(BACKBONES)
run_num = len(done_tags)

for model_tag, model_name in BACKBONES.items():
    if model_tag in done_tags:
        print(f"Skipping {model_tag} (done)")
        continue

    run_num += 1
    print(f"\n>>> Run {run_num}/{total}")

    try:
        result = train_ternary(model_tag, model_name)
        all_results.append(result)
        pd.DataFrame(all_results).to_csv(summary_path, index=False)
        print(f"  Saved. {total - run_num} runs remaining.")
    except Exception as e:
        print(f"  FAILED: {e}")
        if all_results:
            pd.DataFrame(all_results).to_csv(summary_path, index=False)
        if TEMP_TRAIN_DIR.exists():
            shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
        clear_hf_cache()
        raise

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
clear_hf_cache()

print(f"\n{'='*70}")
print(f"All {total} ternary runs complete!")

In [ ]:
# ── Results display ──

results_df = pd.read_csv(summary_path)

display_cols = [
    'model_tag', 'test_accuracy', 'test_macro_f1', 'test_weighted_f1',
    'test_macro_precision', 'test_macro_recall', 'time_seconds'
]
print("="*90)
print("  TERNARY RESULTS — BanglaSarc3 (Test Set)")
print("="*90)
print(results_df[display_cols].sort_values('test_macro_f1', ascending=False).to_string(index=False, float_format='%.4f'))
print()
best = results_df.loc[results_df['test_macro_f1'].idxmax()]
print(f"Best ternary backbone: {best['model_tag']} (Macro-F1={best['test_macro_f1']:.4f})")

In [ ]:
# ── Save artifacts ──

# Confusion matrices
cm_dict = {}
for _, row in results_df.iterrows():
    cm_dict[row['model_tag']] = json.loads(row['confusion_matrix'])
with open(TABLE_DIR / '08_multi_backbone_ternary_confusion_matrices.json', 'w') as f:
    json.dump(cm_dict, f, indent=2)

# Run metadata
meta_cols = ['model_tag', 'model_name', 'dataset', 'train_samples',
             'epochs', 'batch_size', 'lr', 'max_length', 'class_weights', 'time_seconds']
results_df[meta_cols].to_csv(LOG_DIR / '08_multi_backbone_ternary_run_metadata.csv', index=False)

# Classification reports
all_reports = []
for _, row in results_df.iterrows():
    pp = PRED_DIR / f"08_{row['model_tag']}_banglasarc3_ternary_test_predictions.csv"
    if pp.exists():
        pdf = pd.read_csv(pp)
        rpt = classification_report(
            pdf['true_label'], pdf['pred_label'],
            output_dict=True, target_names=LABEL_NAMES
        )
        for cls_name, metrics in rpt.items():
            if isinstance(metrics, dict):
                all_reports.append({
                    'model_tag': row['model_tag'], 'class': cls_name, **metrics
                })
if all_reports:
    pd.DataFrame(all_reports).to_csv(
        REPORT_DIR / '08_multi_backbone_ternary_classification_reports.csv', index=False
    )

print("All artifacts saved.")

In [ ]:
# ── Confusion matrices display ──

for _, row in results_df.iterrows():
    cm = np.array(json.loads(row['confusion_matrix']))
    print(f"\n{row['model_tag']} — Macro-F1: {row['test_macro_f1']:.4f} | Acc: {row['test_accuracy']:.4f}")
    header = '             ' + '  '.join([f'Pred={i:>1}' for i in range(NUM_LABELS)])
    print(header)
    for i in range(NUM_LABELS):
        row_str = f"  True={i}   " + '  '.join([f'{cm[i,j]:>6}' for j in range(NUM_LABELS)])
        print(row_str)

In [ ]:
# ── Per-class F1 comparison (key for thesis) ──

report_path = REPORT_DIR / '08_multi_backbone_ternary_classification_reports.csv'
if report_path.exists():
    rpt_df = pd.read_csv(report_path)
    # Filter to per-class rows only
    per_class = rpt_df[rpt_df['class'].isin(LABEL_NAMES)]
    pivot_f1 = per_class.pivot_table(
        index='model_tag', columns='class', values='f1-score', aggfunc='first'
    )
    pivot_f1 = pivot_f1[LABEL_NAMES]  # Reorder columns
    pivot_f1['macro_avg'] = pivot_f1.mean(axis=1)
    pivot_f1 = pivot_f1.sort_values('macro_avg', ascending=False)
    print("="*80)
    print("  PER-CLASS F1 COMPARISON (Ternary)")
    print("="*80)
    print(pivot_f1.to_string(float_format='%.4f'))
    print()
    weakest_class = pivot_f1[LABEL_NAMES].mean().idxmin()
    print(f"Weakest class across all models: {weakest_class}")

In [ ]:
# ── Final file listing ──

print("Files produced:")
for p in sorted(PROJECT.rglob('08_*')):
    if p.is_file():
        print(f"  {p.relative_to(PROJECT)}  ({p.stat().st_size / 1e6:.1f} MB)")
if os.path.exists('/kaggle/working'):
    print(f"\nDisk free (working): {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"Disk free (tmp):     {disk_free_gb(BIG_TMP):.1f} GB")